---
# 首先，关于卷积的形状变换公式：

### Conv2d 函数：输出形状（输入图形高，宽，卷积核大小，填充，步长）
$$H_{out} = \left\lfloor \frac{H_{in} + 2P - K}{S} \right\rfloor + 1$$

$$W_{out} = \left\lfloor \frac{W_{in} + 2P - K}{S} \right\rfloor + 1$$

其中：
- $H_{in}, W_{in}$: 输入高、宽
- $K$: 卷积核大小（K * K 大小的空间扫描范围）
- $P$: 填充(padding)
- $S$: 步长(stride)

### ConvTranspose2d 输出形状（输入图形高，步长，填充，卷积核大小，输出的填充）
$$H_{out} = (H_{in} - 1) \times S - 2P + K + O_p$$

其中：
- $O_p$: output_padding（补全输出图像尺寸，通常为0）

### 参数数量
- **Conv2d**: $K \times K \times C_{in} \times C_{out} + C_{out}$（有bias时）
- **ConvTranspose2d**: $K \times K \times C_{out} \times C_{in}$（no bias）
- **BatchNorm**: $2 \times C$（gamma和beta）
- **Linear**: $H_{in} \times H_{out} + H_{out}$（有bias时）

In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.nn import Conv2d, ConvTranspose2d, BatchNorm2d, BatchNorm1d, Linear, ReLU, LeakyReLU, Tanh

# 以下内容为AI（Copilot）根据GAN_Test.ipynb，以及关于该文件的问题生成。

---
# 第二部分：Generator 详细分析（基于 DCGAN_Clean.ipynb）

## Generator 工作原理完整流程

### 🎯 核心目标
**从低维隐空间（100维）生成高维图像空间（28×28=784维）**

---

## 📊 完整数据流

```
【输入】隐向量 z ∈ ℝ^100 (从标准正态分布采样)
   ↓
┌─────────────────────────────────────────────────────┐
│ 阶段1: 线性投影 + Reshape                            │
│ 目标: 将1D向量转换为3D特征图                         │
└─────────────────────────────────────────────────────┘
   ↓
[1] Linear(100 → 6272)
    • 参数量: 100 × 6272 = 627,200
    • 输出: (B, 6272)
   ↓
[2] Reshape: (B, 6272) → (B, 128, 7, 7)
    • 将向量重组为 128个 7×7 特征图
    • 这是生成图像的"种子"
   ↓
┌─────────────────────────────────────────────────────┐
│ 阶段2: 第一次上采样 (7×7 → 14×14)                   │
│ 目标: 逐步增大空间分辨率                             │
└─────────────────────────────────────────────────────┘
   ↓
[3] BatchNorm2d(128)
    • 归一化128个通道，稳定训练
    • 参数: 2 × 128 = 256 (γ, β)
   ↓
[4] ReLU(inplace=True)
    • 激活函数: f(x) = max(0, x)
    • 引入非线性，零参数
   ↓
[5] Upsample(scale_factor=2, mode='nearest')
    • 最近邻上采样: 7×7 → 14×14
    • 每个像素复制成 2×2 块
    • 示例: [1,2]  →  [[1,1,2,2],
             [3,4]      [1,1,2,2],
                        [3,3,4,4],
                        [3,3,4,4]]
   ↓
[6] Conv2d(128, 128, kernel=3, padding=1)
    • 卷积平滑上采样的锯齿效果
    • 参数: 3×3×128×128 + 128 = 147,584
    • 输出保持: (B, 128, 14, 14)
   ↓
┌─────────────────────────────────────────────────────┐
│ 阶段3: 第二次上采样 (14×14 → 28×28)                 │
│ 目标: 达到MNIST目标尺寸                              │
└─────────────────────────────────────────────────────┘
   ↓
[7] BatchNorm2d(128)
    • 参数: 256
   ↓
[8] ReLU(inplace=True)
   ↓
[9] Upsample(scale_factor=2)
    • 14×14 → 28×28
   ↓
[10] Conv2d(128, 64, kernel=3, padding=1)
     • 减少通道数: 128 → 64
     • 参数: 3×3×128×64 + 64 = 73,792
     • 输出: (B, 64, 28, 28)
   ↓
┌─────────────────────────────────────────────────────┐
│ 阶段4: 生成最终图像                                  │
│ 目标: 输出单通道灰度图                               │
└─────────────────────────────────────────────────────┘
   ↓
[11] BatchNorm2d(64)
     • 参数: 128
   ↓
[12] ReLU(inplace=True)
   ↓
[13] Conv2d(64, 1, kernel=3, padding=1)
     • 64通道 → 1通道（灰度图）
     • 参数: 3×3×64×1 + 1 = 577
     • 输出: (B, 1, 28, 28)
   ↓
[14] Tanh()
     • 输出范围: [-1, 1]
     • 与数据归一化匹配
   ↓
【输出】生成图像 (B, 1, 28, 28) ∈ [-1, 1]
```

---

## 🔍 关键设计决策

### 1. **为什么用 Upsample + Conv 而不是 ConvTranspose2d？**

**原架构问题 (ConvTranspose2d)**:
- 容易产生"棋盘效应"（checkerboard artifacts）
- 参数效率低
- 训练不稳定

**当前架构优势 (Upsample + Conv)**:
```python
# 旧方案
ConvTranspose2d(128, 128, 4, 2, 1)  # 单步，但有棋盘效应

# 新方案
Upsample(scale_factor=2)           # 先插值放大
Conv2d(128, 128, 3, padding=1)     # 再平滑处理
```

**效果对比**:
- ✅ 更平滑的图像
- ✅ 更少的视觉伪影
- ✅ 训练更稳定

---

### 2. **BatchNorm 的作用**

```
输入分布不稳定 → BatchNorm → 分布标准化 → ReLU效果更好
```

**具体效果**:
- 加速收敛（学习率可以更大）
- 减少对初始化的敏感性
- 起到轻微正则化作用

---

### 3. **为什么选择 Tanh 而不是 Sigmoid？**

| 激活函数 | 输出范围 | 优点 | 缺点 |
|---------|---------|------|------|
| Sigmoid | [0, 1] | 直观 | 梯度消失严重 |
| **Tanh** | **[-1, 1]** | **对称，梯度更好** | **需要数据归一化到[-1,1]** |

**数据归一化匹配**:
```python
# 真实数据归一化
transform = transforms.Normalize((0.5,), (0.5,))  
# 将 [0,1] → [-1,1]

# 生成器输出
output = Tanh(...)  # 输出 [-1,1]
```

---

## 📈 张量形状变化追踪

| 层 | 操作 | 输入形状 | 输出形状 | 参数量 |
|----|------|---------|---------|-------|
| 1 | Linear | (B, 100) | (B, 6272) | 627,200 |
| 2 | Reshape | (B, 6272) | (B, 128, 7, 7) | 0 |
| 3 | BatchNorm2d | (B, 128, 7, 7) | (B, 128, 7, 7) | 256 |
| 4 | ReLU | (B, 128, 7, 7) | (B, 128, 7, 7) | 0 |
| 5 | Upsample | (B, 128, 7, 7) | (B, 128, 14, 14) | 0 |
| 6 | Conv2d | (B, 128, 14, 14) | (B, 128, 14, 14) | 147,584 |
| 7 | BatchNorm2d | (B, 128, 14, 14) | (B, 128, 14, 14) | 256 |
| 8 | ReLU | (B, 128, 14, 14) | (B, 128, 14, 14) | 0 |
| 9 | Upsample | (B, 128, 14, 14) | (B, 128, 28, 28) | 0 |
| 10 | Conv2d | (B, 128, 28, 28) | (B, 64, 28, 28) | 73,792 |
| 11 | BatchNorm2d | (B, 64, 28, 28) | (B, 64, 28, 28) | 128 |
| 12 | ReLU | (B, 64, 28, 28) | (B, 64, 28, 28) | 0 |
| 13 | Conv2d | (B, 64, 28, 28) | (B, 1, 28, 28) | 577 |
| 14 | Tanh | (B, 1, 28, 28) | (B, 1, 28, 28) | 0 |

**总参数量**: 627,200 + 147,584 + 73,792 + 577 + 640 = **849,793**

---

## 🎨 直观理解：从噪声到图像

```
步骤1: 隐向量 z (100个随机数)
   [0.23, -1.45, 0.88, ..., -0.34]
   ↓ Linear 线性投影
   
步骤2: 展开为小特征图 (128个 7×7)
   ┌─────┬─────┬─────┐
   │ F1  │ F2  │ ... │ 128个特征图
   │ 7×7 │ 7×7 │     │ 每个7×7像素
   └─────┴─────┴─────┘
   ↓ 归一化 + 激活 + 上采样
   
步骤3: 第一次放大 (128个 14×14)
   ┌──────┬──────┐
   │ 模糊 │ 边缘 │ 学习不同特征
   │ 14×14│ 14×14│ 的抽象表示
   └──────┴──────┘
   ↓ 继续上采样
   
步骤4: 第二次放大 (64个 28×28)
   ┌────────┬────────┐
   │ 笔画1  │ 笔画2  │ 更具体的
   │ 28×28  │ 28×28  │ 视觉特征
   └────────┴────────┘
   ↓ 融合为单通道
   
步骤5: 最终图像 (1个 28×28)
   ┌──────────┐
   │    8     │ 完整的
   │  ████    │ 数字图像
   │    ██    │ 
   └──────────┘
```

---

## ⚙️ 代码实现对应

```python
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        # 阶段1: 线性投影
        self.fc = nn.Linear(cfg.latent_dim, cfg.g_features * 7 * 7)
        
        self.main = nn.Sequential(
            # 阶段2: 第一次上采样块 (7→14)
            nn.BatchNorm2d(cfg.g_features),
            nn.ReLU(True),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(cfg.g_features, cfg.g_features, 3, padding=1),
            
            # 阶段3: 第二次上采样块 (14→28)
            nn.BatchNorm2d(cfg.g_features),
            nn.ReLU(True),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(cfg.g_features, cfg.g_features // 2, 3, padding=1),
            
            # 阶段4: 输出层
            nn.BatchNorm2d(cfg.g_features // 2),
            nn.ReLU(True),
            nn.Conv2d(cfg.g_features // 2, cfg.channels, 3, padding=1),
            nn.Tanh()
        )
    
    def forward(self, z):
        # 步骤1: 线性投影
        x = self.fc(z)                                    # (B,100) → (B,6272)
        # 步骤2: Reshape为特征图
        x = x.view(x.size(0), cfg.g_features, 7, 7)      # → (B,128,7,7)
        # 步骤3-5: 通过卷积块逐步上采样
        return self.main(x)                               # → (B,1,28,28)
```

---

## 🧠 核心原理总结

1. **降维到升维的映射**
   - 输入: 100维压缩表示
   - 输出: 784维图像
   - 通过学习找到最优映射函数

2. **渐进式生成**
   - 不是一步到位生成28×28
   - 而是 7×7 → 14×14 → 28×28 逐步细化
   - 类似画家：先画草图，再添细节

3. **特征层次结构**
   - 7×7: 全局结构（是什么数字）
   - 14×14: 局部形状（笔画方向）
   - 28×28: 细节纹理（边缘平滑度）

4. **对抗训练中的角色**
   ```
   Generator 目标: max E[log(D(G(z)))]
   翻译: 生成让判别器认为是真实的图像
   策略: 学习真实数据的分布特征
   ```

In [3]:
# =====================================================
# 层 1-2: 隐向量 → 初始特征图
# =====================================================

print("\n" + "="*80)
print("层 1-2: 隐向量 → 初始特征图")
print("="*80)

print("\n[层1] Linear层")
print("-" * 80)
print("功能: 将1D隐向量映射到更高维空间")
print()
print("配置:")
print("  输入维度: 100 (latent_dim)")
print("  输出维度: 128 × 7 × 7 = 6,272 (g_features × 7 × 7)")
print("  bias: True")
print()
print("计算细节:")
print("  参数数量 = 100 × 6272 + 6272 = 627,200 + 6,272 = 633,472")
print()
print("输入/输出:")
batch_size = 2  # 演示用小batch
z = torch.randn(batch_size, 100)
print(f"  输入形状: {z.shape}  (batch_size=2, latent_dim=100)")
print(f"  输入示例值范围: [{z.min():.4f}, {z.max():.4f}]")

fc_layer = Linear(100, 128*7*7)
out_fc = fc_layer(z)
print(f"  输出形状: {out_fc.shape}  (batch_size=2, 6272)")
print(f"  输出示例值范围: [{out_fc.min():.4f}, {out_fc.max():.4f}]")

print("\n[层2] Reshape到4D张量")
print("-" * 80)
print("功能: 为卷积操作做准备")
print()
out_reshape = out_fc.view(batch_size, 128, 7, 7)
print(f"  输入形状: {out_fc.shape}  (flattened)")
print(f"  输出形状: {out_reshape.shape}  (batch, channels, height, width)")
print(f"  说明: 从 (2, 6272) → (2, 128, 7, 7)")
print(f"       128个特征图，每个大小7×7像素")
print()
print(f"  这是生成过程的'种子'特征图")
print(f"  后续将通过上采样放大到28×28")


层 1-2: 隐向量 → 初始特征图

[层1] Linear层
--------------------------------------------------------------------------------
功能: 将1D隐向量映射到更高维空间

配置:
  输入维度: 100 (latent_dim)
  输出维度: 128 × 7 × 7 = 6,272 (g_features × 7 × 7)
  bias: True

计算细节:
  参数数量 = 100 × 6272 + 6272 = 627,200 + 6,272 = 633,472

输入/输出:
  输入形状: torch.Size([2, 100])  (batch_size=2, latent_dim=100)
  输入示例值范围: [-3.0173, 3.0003]
  输出形状: torch.Size([2, 6272])  (batch_size=2, 6272)
  输出示例值范围: [-2.4166, 2.3121]

[层2] Reshape到4D张量
--------------------------------------------------------------------------------
功能: 为卷积操作做准备

  输入形状: torch.Size([2, 6272])  (flattened)
  输出形状: torch.Size([2, 128, 7, 7])  (batch, channels, height, width)
  说明: 从 (2, 6272) → (2, 128, 7, 7)
       128个特征图，每个大小7×7像素

  这是生成过程的'种子'特征图
  后续将通过上采样放大到28×28


In [4]:
# =====================================================
# 第3-6层：第一次上采样块 7x7 → 14x14
# =====================================================

print("\n" + "="*80)
print("层 3-6: 第一次上采样块 (7×7 → 14×14)")
print("="*80)

print("\n[层3] BatchNorm2d")
print("-" * 80)
print("功能: 归一化特征图，稳定训练")
print()
print("配置:")
print("  通道数: 128")
print("  参数数量: 2 × 128 = 256 (gamma和beta)")
print()
batch_norm_1 = BatchNorm2d(128)
out_bn_1 = batch_norm_1(out_reshape)
print(f"  输入形状: {out_reshape.shape}")
print(f"  输出形状: {out_bn_1.shape}")
print(f"  输出示例值范围: [{out_bn_1.min():.4f}, {out_bn_1.max():.4f}]")
print(f"  说明: BatchNorm2d对每个通道独立归一化，使分布标准化")

print("\n[层4] ReLU激活")
print("-" * 80)
print("功能: 非线性激活，引入表现力")
print()
print("公式: y = max(0, x)")
print()
relu = ReLU(inplace=True)
out_relu_1 = torch.relu(out_bn_1)
print(f"  输入形状: {out_bn_1.shape}")
print(f"  输出形状: {out_relu_1.shape}")
print(f"  输出示例值范围: [{out_relu_1.min():.4f}, {out_relu_1.max():.4f}]")
print(f"  说明: 所有负值变为0，正值保留")

print("\n[层5] Upsample (最近邻上采样)")
print("-" * 80)
print("功能: 放大特征图空间尺寸")
print()
print("配置:")
print("  scale_factor: 2 (放大2倍)")
print("  mode: 'nearest' (最近邻插值，默认)")
print()
print("工作原理:")
print("  每个像素值复制成2×2的块")
print("  示例: [[1,2],  →  [[1,1,2,2],")
print("         [3,4]]      [1,1,2,2],")
print("                     [3,3,4,4],")
print("                     [3,3,4,4]]")
print()
upsample_1 = nn.Upsample(scale_factor=2, mode='nearest')
out_up_1 = upsample_1(out_relu_1)
print(f"  输入形状: {out_relu_1.shape}  (2, 128, 7, 7)")
print(f"  输出形状: {out_up_1.shape}  (2, 128, 14, 14)")
print(f"  输出示例值范围: [{out_up_1.min():.4f}, {out_up_1.max():.4f}]")
print(f"  参数数量: 0 (无需训练的参数)")
print()
print("  优势: 相比ConvTranspose2d，避免了'棋盘效应'")

print("\n[层6] Conv2d (平滑卷积)")
print("-" * 80)
print("功能: 平滑上采样产生的锯齿，保持通道数")
print()
print("配置:")
print("  输入通道: 128")
print("  输出通道: 128")
print("  卷积核大小: 3×3")
print("  步长 (stride): 1")
print("  填充 (padding): 1")
print("  bias: True")
print()
print("形状计算:")
print("  H_out = (H_in + 2×padding - kernel) / stride + 1")
print("  H_out = (14 + 2×1 - 3) / 1 + 1 = 14 (保持不变)")
print()
print("参数数量:")
print("  = 3 × 3 × 128 × 128 + 128 = 147,456 + 128 = 147,584")
print()
conv_1 = Conv2d(128, 128, 3, padding=1)
out_conv_1 = conv_1(out_up_1)
print(f"  输入形状: {out_up_1.shape}  (2, 128, 14, 14)")
print(f"  输出形状: {out_conv_1.shape}  (2, 128, 14, 14)")
print(f"  输出示例值范围: [{out_conv_1.min():.4f}, {out_conv_1.max():.4f}]")
print(f"  总参数: {sum(p.numel() for p in conv_1.parameters()):,}")
print()
print("  说明: 这层学习如何平滑上采样的结果，减少视觉伪影")


层 3-6: 第一次上采样块 (7×7 → 14×14)

[层3] BatchNorm2d
--------------------------------------------------------------------------------
功能: 归一化特征图，稳定训练

配置:
  通道数: 128
  参数数量: 2 × 128 = 256 (gamma和beta)

  输入形状: torch.Size([2, 128, 7, 7])
  输出形状: torch.Size([2, 128, 7, 7])
  输出示例值范围: [-3.8395, 3.7416]
  说明: BatchNorm2d对每个通道独立归一化，使分布标准化

[层4] ReLU激活
--------------------------------------------------------------------------------
功能: 非线性激活，引入表现力

公式: y = max(0, x)

  输入形状: torch.Size([2, 128, 7, 7])
  输出形状: torch.Size([2, 128, 7, 7])
  输出示例值范围: [0.0000, 3.7416]
  说明: 所有负值变为0，正值保留

[层5] Upsample (最近邻上采样)
--------------------------------------------------------------------------------
功能: 放大特征图空间尺寸

配置:
  scale_factor: 2 (放大2倍)
  mode: 'nearest' (最近邻插值，默认)

工作原理:
  每个像素值复制成2×2的块
  示例: [[1,2],  →  [[1,1,2,2],
         [3,4]]      [1,1,2,2],
                     [3,3,4,4],
                     [3,3,4,4]]

  输入形状: torch.Size([2, 128, 7, 7])  (2, 128, 7, 7)
  输出形状: torch.Size([2, 128, 14, 14])  (2, 1

In [6]:
# =====================================================
# 第7-10层：第二次上采样块 14x14 → 28x28
# =====================================================

print("\n" + "="*80)
print("层 7-10: 第二次上采样块 (14×14 → 28×28)")
print("="*80)

print("\n[层7] BatchNorm2d")
print("-" * 80)
print("功能: 归一化卷积层输出")
print()
print("配置:")
print("  通道数: 128")
print("  参数数量: 2 × 128 = 256")
print()
batch_norm_2 = BatchNorm2d(128)
out_bn_2 = batch_norm_2(out_conv_1)
print(f"  输入形状: {out_conv_1.shape}")
print(f"  输出形状: {out_bn_2.shape}")
print(f"  输出示例值范围: [{out_bn_2.min():.4f}, {out_bn_2.max():.4f}]")

print("\n[层8] ReLU激活")
print("-" * 80)
out_relu_2 = torch.relu(out_bn_2)
print(f"  输入形状: {out_bn_2.shape}")
print(f"  输出形状: {out_relu_2.shape}")
print(f"  输出示例值范围: [{out_relu_2.min():.4f}, {out_relu_2.max():.4f}]")

print("\n[层9] Upsample (第二次上采样)")
print("-" * 80)
print("功能: 进一步放大到目标尺寸")
print()
print("配置:")
print("  scale_factor: 2")
print("  14×14 → 28×28 (MNIST目标尺寸)")
print()
upsample_2 = nn.Upsample(scale_factor=2, mode='nearest')
out_up_2 = upsample_2(out_relu_2)
print(f"  输入形状: {out_relu_2.shape}  (2, 128, 14, 14)")
print(f"  输出形状: {out_up_2.shape}  (2, 128, 28, 28)")
print(f"  输出示例值范围: [{out_up_2.min():.4f}, {out_up_2.max():.4f}]")
print(f"  说明: 已达到MNIST的目标空间尺寸28×28")

print("\n[层10] Conv2d (通道减半)")
print("-" * 80)
print("功能: 平滑上采样 + 减少通道数")
print()
print("配置:")
print("  输入通道: 128")
print("  输出通道: 64  (通道减半)")
print("  卷积核大小: 3×3")
print("  步长: 1, 填充: 1")
print()
print("参数数量:")
print("  = 3 × 3 × 128 × 64 + 64 = 73,728 + 64 = 73,792")
print()
conv_2 = Conv2d(128, 64, 3, padding=1)
out_conv_2 = conv_2(out_up_2)
print(f"  输入形状: {out_up_2.shape}  (2, 128, 28, 28)")
print(f"  输出形状: {out_conv_2.shape}  (2, 64, 28, 28)")
print(f"  输出示例值范围: [{out_conv_2.min():.4f}, {out_conv_2.max():.4f}]")
print(f"  总参数: {sum(p.numel() for p in conv_2.parameters()):,}")
print()
print("  说明: 通道数递减(128→64)，为最终输出做准备")


层 7-10: 第二次上采样块 (14×14 → 28×28)

[层7] BatchNorm2d
--------------------------------------------------------------------------------
功能: 归一化卷积层输出

配置:
  通道数: 128
  参数数量: 2 × 128 = 256

  输入形状: torch.Size([2, 128, 14, 14])
  输出形状: torch.Size([2, 128, 14, 14])
  输出示例值范围: [-4.4459, 4.2167]

[层8] ReLU激活
--------------------------------------------------------------------------------
  输入形状: torch.Size([2, 128, 14, 14])
  输出形状: torch.Size([2, 128, 14, 14])
  输出示例值范围: [0.0000, 4.2167]

[层9] Upsample (第二次上采样)
--------------------------------------------------------------------------------
功能: 进一步放大到目标尺寸

配置:
  scale_factor: 2
  14×14 → 28×28 (MNIST目标尺寸)

  输入形状: torch.Size([2, 128, 14, 14])  (2, 128, 14, 14)
  输出形状: torch.Size([2, 128, 28, 28])  (2, 128, 28, 28)
  输出示例值范围: [0.0000, 4.2167]
  说明: 已达到MNIST的目标空间尺寸28×28

[层10] Conv2d (通道减半)
--------------------------------------------------------------------------------
功能: 平滑上采样 + 减少通道数

配置:
  输入通道: 128
  输出通道: 64  (通道减半)
  卷积核大小: 3×3
  步长: 1, 填充

In [7]:
# =====================================================
# 第11-14层：输出层生成最终图像
# =====================================================

print("\n" + "="*80)
print("层 11-14: 输出层 - 生成最终图像")
print("="*80)

print("\n[层11] BatchNorm2d")
print("-" * 80)
print("功能: 最后一次归一化")
print()
print("配置:")
print("  通道数: 64")
print("  参数数量: 2 × 64 = 128")
print()
batch_norm_3 = BatchNorm2d(64)
out_bn_3 = batch_norm_3(out_conv_2)
print(f"  输入形状: {out_conv_2.shape}")
print(f"  输出形状: {out_bn_3.shape}")
print(f"  输出示例值范围: [{out_bn_3.min():.4f}, {out_bn_3.max():.4f}]")

print("\n[层12] ReLU激活")
print("-" * 80)
out_relu_3 = torch.relu(out_bn_3)
print(f"  输入形状: {out_bn_3.shape}")
print(f"  输出形状: {out_relu_3.shape}")
print(f"  输出示例值范围: [{out_relu_3.min():.4f}, {out_relu_3.max():.4f}]")

print("\n[层13] Conv2d (最后一层卷积)")
print("-" * 80)
print("功能: 生成单通道灰度图像")
print()
print("配置:")
print("  输入通道: 64")
print("  输出通道: 1  (灰度图)")
print("  卷积核大小: 3×3")
print("  步长: 1, 填充: 1 (保持尺寸)")
print()
print("参数数量:")
print("  = 3 × 3 × 64 × 1 + 1 = 576 + 1 = 577")
print()
conv_final = Conv2d(64, 1, 3, padding=1)
out_conv_final = conv_final(out_relu_3)
print(f"  输入形状: {out_relu_3.shape}  (2, 64, 28, 28)")
print(f"  输出形状: {out_conv_final.shape}  (2, 1, 28, 28)")
print(f"  输出示例值范围: [{out_conv_final.min():.4f}, {out_conv_final.max():.4f}]")
print(f"  总参数: {sum(p.numel() for p in conv_final.parameters()):,}")
print()
print("  说明: 64个特征图融合成1个灰度图像!")

print("\n[层14] Tanh激活 (输出层)")
print("-" * 80)
print("功能: 将输出映射到[-1, 1]范围")
print()
print("公式: y = tanh(x) = (e^x - e^-x) / (e^x + e^-x)")
print()
print("为什么用Tanh?")
print("  • 输出范围[-1, 1]对称，适合图像生成")
print("  • 与数据归一化匹配: Normalize((0.5,), (0.5,))")
print("  • 比Sigmoid梯度更好，训练更稳定")
print()
tanh = Tanh()
out_tanh = tanh(out_conv_final)
print(f"  输入形状: {out_conv_final.shape}")
print(f"  输出形状: {out_tanh.shape}")
print(f"  输出值范围: [{out_tanh.min():.4f}, {out_tanh.max():.4f}]")
print(f"  说明: 所有值现在严格在[-1, 1]范围内")
print()
print("  ✓ 最终生成图像完成! (B, 1, 28, 28) ∈ [-1, 1]")


层 11-14: 输出层 - 生成最终图像

[层11] BatchNorm2d
--------------------------------------------------------------------------------
功能: 最后一次归一化

配置:
  通道数: 64
  参数数量: 2 × 64 = 128

  输入形状: torch.Size([2, 64, 28, 28])
  输出形状: torch.Size([2, 64, 28, 28])
  输出示例值范围: [-4.3892, 4.8309]

[层12] ReLU激活
--------------------------------------------------------------------------------
  输入形状: torch.Size([2, 64, 28, 28])
  输出形状: torch.Size([2, 64, 28, 28])
  输出示例值范围: [0.0000, 4.8309]

[层13] Conv2d (最后一层卷积)
--------------------------------------------------------------------------------
功能: 生成单通道灰度图像

配置:
  输入通道: 64
  输出通道: 1  (灰度图)
  卷积核大小: 3×3
  步长: 1, 填充: 1 (保持尺寸)

参数数量:
  = 3 × 3 × 64 × 1 + 1 = 576 + 1 = 577

  输入形状: torch.Size([2, 64, 28, 28])  (2, 64, 28, 28)
  输出形状: torch.Size([2, 1, 28, 28])  (2, 1, 28, 28)
  输出示例值范围: [-1.0309, 1.1032]
  总参数: 577

  说明: 64个特征图融合成1个灰度图像!

[层14] Tanh激活 (输出层)
--------------------------------------------------------------------------------
功能: 将输出映射到[-1, 1]范围

公式: y = t

---
# 第三部分：Generator 总体统计

## Generator 的完整路径

In [8]:
# Generator完整总结
print("\n" + "="*80)
print("Generator 完整汇总 (基于 DCGAN_Clean.ipynb)")
print("="*80)

print("\n架构: Upsample + Conv2d (避免棋盘效应)")
print("输入: 隐向量 z (batch_size, 100)")
print("输出: 生成的图像 (batch_size, 1, 28, 28) ∈ [-1, 1]")
print()

print("\n层级映射关系:")
print()
stages = [
    ("1. Linear", "(B, 100)", "(B, 6272)", 627200),
    ("2. Reshape", "(B, 6272)", "(B, 128, 7, 7)", 0),
    ("3. BatchNorm2d", "(B, 128, 7, 7)", "(B, 128, 7, 7)", 256),
    ("4. ReLU", "(B, 128, 7, 7)", "(B, 128, 7, 7)", 0),
    ("5. Upsample ×2", "(B, 128, 7, 7)", "(B, 128, 14, 14)", 0),
    ("6. Conv2d", "(B, 128, 14, 14)", "(B, 128, 14, 14)", 147584),
    ("7. BatchNorm2d", "(B, 128, 14, 14)", "(B, 128, 14, 14)", 256),
    ("8. ReLU", "(B, 128, 14, 14)", "(B, 128, 14, 14)", 0),
    ("9. Upsample ×2", "(B, 128, 14, 14)", "(B, 128, 28, 28)", 0),
    ("10. Conv2d 128→64", "(B, 128, 28, 28)", "(B, 64, 28, 28)", 73792),
    ("11. BatchNorm2d", "(B, 64, 28, 28)", "(B, 64, 28, 28)", 128),
    ("12. ReLU", "(B, 64, 28, 28)", "(B, 64, 28, 28)", 0),
    ("13. Conv2d 64→1", "(B, 64, 28, 28)", "(B, 1, 28, 28)", 577),
    ("14. Tanh", "(B, 1, 28, 28)", "(B, 1, 28, 28) ∈ [-1,1]", 0),
]

print(f"{'序号':<18} {'输入形状':<22} {'输出形状':<25} {'参数数量':<15}")
print("-" * 80)
total_params = 0
for stage, input_shape, output_shape, params in stages:
    print(f"{stage:<18} {input_shape:<22} {output_shape:<25} {params:>14,}")
    total_params += params

print("-" * 80)
print(f"{'总计':<18} {'':<22} {'':<25} {total_params:>14,}")
print()

print("\n关键特征:")
print("  • 隐向量维度: 100")
print("  • 初始特征图: 128 通道 × 7×7 分辨率 = 6,272 个值")
print("  • 两次上采样: 7×7 → 14×14 → 28×28")
print("  • 通道递减: 128 → 128 → 64 → 1")
print("  • 上采样方式: Upsample(最近邻) + Conv2d(平滑)")
print("  • 最终输出: 1×28×28 灰度图像，值在[-1, 1]")
print(f"  • 总参数数: {total_params:,}")
print()
print("\n架构优势 (相比ConvTranspose2d):")
print("  ✓ 避免棋盘效应 (checkerboard artifacts)")
print("  ✓ 参数效率更高 (849,793 vs 3,572,864)")
print("  ✓ 训练更稳定")
print("  ✓ 生成图像更平滑")
print()
print("\n数据流总结:")
print("  [隐空间] 100维随机向量")
print("     ↓ Linear投影")
print("  [特征图] 128×7×7 (种子特征)")
print("     ↓ Upsample块1 (BN+ReLU+Up+Conv)")
print("  [放大1] 128×14×14")
print("     ↓ Upsample块2 (BN+ReLU+Up+Conv)")
print("  [放大2] 64×28×28")
print("     ↓ 输出层 (BN+ReLU+Conv+Tanh)")
print("  [图像] 1×28×28 ∈ [-1,1]")


Generator 完整汇总 (基于 DCGAN_Clean.ipynb)

架构: Upsample + Conv2d (避免棋盘效应)
输入: 隐向量 z (batch_size, 100)
输出: 生成的图像 (batch_size, 1, 28, 28) ∈ [-1, 1]


层级映射关系:

序号                 输入形状                   输出形状                      参数数量           
--------------------------------------------------------------------------------
1. Linear          (B, 100)               (B, 6272)                        627,200
2. Reshape         (B, 6272)              (B, 128, 7, 7)                         0
3. BatchNorm2d     (B, 128, 7, 7)         (B, 128, 7, 7)                       256
4. ReLU            (B, 128, 7, 7)         (B, 128, 7, 7)                         0
5. Upsample ×2     (B, 128, 7, 7)         (B, 128, 14, 14)                       0
6. Conv2d          (B, 128, 14, 14)       (B, 128, 14, 14)                 147,584
7. BatchNorm2d     (B, 128, 14, 14)       (B, 128, 14, 14)                     256
8. ReLU            (B, 128, 14, 14)       (B, 128, 14, 14)                       0
9. Upsample ×2   

In [9]:
# =====================================================
# Upsample + Conv2d 的工作原理演示
# =====================================================

print("\n" + "="*80)
print("Upsample + Conv2d 工作原理对比")
print("="*80)

print("\n演示: 为什么用 Upsample+Conv 而不是 ConvTranspose2d?")
print()

# 创建一个简单的2×2输入
input_2x2 = torch.tensor([[[[1., 2.],
                             [3., 4.]]]])

print("原始输入 (1×1×2×2):")
print(input_2x2.squeeze())
print()

# 方法1: Upsample (最近邻)
print("方法1: Upsample(scale_factor=2, mode='nearest')")
print("-" * 80)
upsample_nearest = nn.Upsample(scale_factor=2, mode='nearest')
out_nearest = upsample_nearest(input_2x2)
print(f"输出形状: {out_nearest.shape}")
print("输出值 (4×4):")
print(out_nearest.squeeze())
print()
print("特点: 简单复制，产生'块状'效果")
print("      [[1,2],   →   [[1,1,2,2],")
print("       [3,4]]         [1,1,2,2],")
print("                      [3,3,4,4],")
print("                      [3,3,4,4]]")
print()

# 方法2: Upsample (双线性)
print("方法2: Upsample(scale_factor=2, mode='bilinear')")
print("-" * 80)
upsample_bilinear = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
out_bilinear = upsample_bilinear(input_2x2)
print(f"输出形状: {out_bilinear.shape}")
print("输出值 (4×4):")
print(out_bilinear.squeeze())
print()
print("特点: 线性插值，更平滑但略模糊")
print()

# 方法3: Upsample + Conv2d (DCGAN_Clean使用的方法)
print("方法3: Upsample + Conv2d (DCGAN_Clean 使用)")
print("-" * 80)
print("步骤1: 先用Upsample放大")
out_up = upsample_nearest(input_2x2)
print(f"  Upsample后: {out_up.shape}")
print()
print("步骤2: 再用Conv2d平滑和学习特征")
conv_smooth = Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
# 设置一个简单的平滑卷积核
with torch.no_grad():
    conv_smooth.weight.fill_(1/9)  # 平均池化核
out_smooth = conv_smooth(out_up)
print(f"  Conv2d后: {out_smooth.shape}")
print("  输出值 (平滑后):")
print(out_smooth.squeeze())
print()
print("优势:")
print("  ✓ 先放大(Upsample) 再平滑(Conv)，两步分离")
print("  ✓ Conv2d可以学习如何最优地平滑和提取特征")
print("  ✓ 避免ConvTranspose2d的棋盘效应")
print("  ✓ 参数更少，训练更稳定")
print()

# 对比 ConvTranspose2d
print("对比: ConvTranspose2d (旧方法)")
print("-" * 80)
conv_transpose = nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1, bias=False)
with torch.no_grad():
    conv_transpose.weight.fill_(0.25)
out_convt = conv_transpose(input_2x2)
print(f"输出形状: {out_convt.shape}")
print("输出值:")
print(out_convt.squeeze())
print()
print("问题:")
print("  ✗ 容易产生棋盘效应 (checkerboard artifacts)")
print("  ✗ 参数多但效率低")
print("  ✗ 重叠区域会有不均匀的覆盖")
print()

print("="*80)
print("结论: Upsample+Conv 是更好的上采样方案!")
print("="*80)


Upsample + Conv2d 工作原理对比

演示: 为什么用 Upsample+Conv 而不是 ConvTranspose2d?

原始输入 (1×1×2×2):
tensor([[1., 2.],
        [3., 4.]])

方法1: Upsample(scale_factor=2, mode='nearest')
--------------------------------------------------------------------------------
输出形状: torch.Size([1, 1, 4, 4])
输出值 (4×4):
tensor([[1., 1., 2., 2.],
        [1., 1., 2., 2.],
        [3., 3., 4., 4.],
        [3., 3., 4., 4.]])

特点: 简单复制，产生'块状'效果
      [[1,2],   →   [[1,1,2,2],
       [3,4]]         [1,1,2,2],
                      [3,3,4,4],
                      [3,3,4,4]]

方法2: Upsample(scale_factor=2, mode='bilinear')
--------------------------------------------------------------------------------
输出形状: torch.Size([1, 1, 4, 4])
输出值 (4×4):
tensor([[1.0000, 1.2500, 1.7500, 2.0000],
        [1.5000, 1.7500, 2.2500, 2.5000],
        [2.5000, 2.7500, 3.2500, 3.5000],
        [3.0000, 3.2500, 3.7500, 4.0000]])

特点: 线性插值，更平滑但略模糊

方法3: Upsample + Conv2d (DCGAN_Clean 使用)
---------------------------------------------------

---
## Generator 完整示例：从噪声到图像

下面的代码展示了一个完整的生成过程

In [10]:
# =====================================================
# 完整Generator模拟 (基于DCGAN_Clean架构)
# =====================================================

print("\n" + "="*80)
print("完整Generator前向传播示例")
print("="*80)

# 配置
class SimpleConfig:
    latent_dim = 100
    g_features = 128
    channels = 1

cfg = SimpleConfig()

# 构建完整的Generator
class SimpleGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(cfg.latent_dim, cfg.g_features * 7 * 7)
        
        self.main = nn.Sequential(
            # 第一次上采样块 7→14
            nn.BatchNorm2d(cfg.g_features),
            nn.ReLU(True),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(cfg.g_features, cfg.g_features, 3, padding=1),
            
            # 第二次上采样块 14→28
            nn.BatchNorm2d(cfg.g_features),
            nn.ReLU(True),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(cfg.g_features, cfg.g_features // 2, 3, padding=1),
            
            # 输出层
            nn.BatchNorm2d(cfg.g_features // 2),
            nn.ReLU(True),
            nn.Conv2d(cfg.g_features // 2, cfg.channels, 3, padding=1),
            nn.Tanh()
        )
    
    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), cfg.g_features, 7, 7)
        return self.main(x)

# 实例化
G_demo = SimpleGenerator()
total_params = sum(p.numel() for p in G_demo.parameters())

print(f"\n构建Generator成功!")
print(f"总参数数: {total_params:,}")
print()

# 生成一个样本
print("生成示例:")
print("-" * 80)
z_demo = torch.randn(1, 100)  # 1个样本
print(f"输入: 隐向量 z")
print(f"  形状: {z_demo.shape}")
print(f"  值范围: [{z_demo.min():.4f}, {z_demo.max():.4f}]")
print(f"  前5个值: {z_demo[0, :5].tolist()}")
print()

# 前向传播
with torch.no_grad():
    G_demo.eval()
    generated_img = G_demo(z_demo)

print(f"输出: 生成图像")
print(f"  形状: {generated_img.shape}")
print(f"  值范围: [{generated_img.min():.4f}, {generated_img.max():.4f}]")
print(f"  数据类型: {generated_img.dtype}")
print()

print("逐层形状变化追踪:")
print("-" * 80)

# 手动追踪每一步
with torch.no_grad():
    x = G_demo.fc(z_demo)
    print(f"1. Linear:        {z_demo.shape} → {x.shape}")
    
    x = x.view(1, cfg.g_features, 7, 7)
    print(f"2. Reshape:       (1, 6272) → {x.shape}")
    
    # 模拟每层
    layer_idx = 3
    for i, layer in enumerate(G_demo.main):
        x = layer(x)
        layer_name = layer.__class__.__name__
        print(f"{layer_idx}. {layer_name:<15} → {x.shape}")
        layer_idx += 1

print()
print("="*80)
print("✓ Generator 完整流程演示完成!")
print("="*80)
print()
print("总结:")
print("  • 从 100维随机噪声 开始")
print("  • 经过 14层网络 处理")
print("  • 最终生成 28×28 灰度图像")
print("  • 输出范围 [-1, 1] 适配数据归一化")
print(f"  • 总共使用 {total_params:,} 个可训练参数")


完整Generator前向传播示例

构建Generator成功!
总参数数: 856,065

生成示例:
--------------------------------------------------------------------------------
输入: 隐向量 z
  形状: torch.Size([1, 100])
  值范围: [-2.8996, 2.1651]
  前5个值: [-0.1355094462633133, 0.7020117044448853, -0.33404698967933655, -0.5073495507240295, -1.482458472251892]

输出: 生成图像
  形状: torch.Size([1, 1, 28, 28])
  值范围: [-0.0725, 0.1118]
  数据类型: torch.float32

逐层形状变化追踪:
--------------------------------------------------------------------------------
1. Linear:        torch.Size([1, 100]) → torch.Size([1, 6272])
2. Reshape:       (1, 6272) → torch.Size([1, 128, 7, 7])
3. BatchNorm2d     → torch.Size([1, 128, 7, 7])
4. ReLU            → torch.Size([1, 128, 7, 7])
5. Upsample        → torch.Size([1, 128, 14, 14])
6. Conv2d          → torch.Size([1, 128, 14, 14])
7. BatchNorm2d     → torch.Size([1, 128, 14, 14])
8. ReLU            → torch.Size([1, 128, 14, 14])
9. Upsample        → torch.Size([1, 128, 28, 28])
10. Conv2d          → torch.Size([1, 64,

---
# 第四部分：Discriminator 详细分析 (基于 DCGAN_Clean.ipynb)

## Discriminator 架构概览

```
输入: 图像 (1, 28, 28) ∈ [-1, 1]
  ↓
┌─────────────────────────────────────────────────────┐
│ 第一层（不使用BatchNorm）                             │
└─────────────────────────────────────────────────────┘
[1] Conv2d: (1, 28, 28) → (64, 14, 14)
[2] LeakyReLU(0.2): 激活
[3] Dropout2d(0.3): 防止过拟合
  ↓
┌─────────────────────────────────────────────────────┐
│ 第一个卷积块                                          │
└─────────────────────────────────────────────────────┘
[4] Conv2d: (64, 14, 14) → (128, 7, 7)
[5] BatchNorm2d: 归一化
[6] LeakyReLU(0.2): 激活
[7] Dropout2d(0.3): 防止过拟合
  ↓
┌─────────────────────────────────────────────────────┐
│ 第二个卷积块                                          │
└─────────────────────────────────────────────────────┘
[8] Conv2d: (128, 7, 7) → (256, 3, 3)
[9] BatchNorm2d: 归一化
[10] LeakyReLU(0.2): 激活
[11] Dropout2d(0.3): 防止过拟合
  ↓
┌─────────────────────────────────────────────────────┐
│ 第三个卷积块（最后的特征提取）                         │
└─────────────────────────────────────────────────────┘
[12] Conv2d: (256, 3, 3) → (512, 1, 1)
[13] BatchNorm2d: 归一化
[14] LeakyReLU(0.2): 激活
  ↓
┌─────────────────────────────────────────────────────┐
│ 分类层                                                │
└─────────────────────────────────────────────────────┘
[15] Flatten: (512, 1, 1) → (512,)
[16] Linear: (512) → (1)
[17] Sigmoid: 输出概率 [0, 1]
  ↓
输出: 判别概率 ∈ [0, 1]
     0 = 假图像，1 = 真图像
```

---

## 关键设计决策

### 1. 为什么第一层不用 BatchNorm？
- 保留真实数据的原始统计特征
- 让判别器直接学习输入分布
- DCGAN论文的标准实践

### 2. 为什么用 LeakyReLU 而不是 ReLU？
| 激活函数 | 负值处理 | 优点 | 在判别器中 |
|---------|---------|------|-----------|
| ReLU | 截断为0 | 简单 | 可能导致"神经元死亡" |
| **LeakyReLU(0.2)** | **乘以0.2** | **梯度流动更好** | **推荐使用** |

### 3. Dropout2d 的作用
- 训练时随机丢弃整个特征图（通道）
- 防止判别器过度拟合训练数据
- dropout=0.3 表示30%的特征图被丢弃

### 4. 输出 Sigmoid
- 将 logits 映射到 [0, 1] 概率
- 0.0 = 完全确定是假的
- 0.5 = 不确定
- 1.0 = 完全确定是真的

In [12]:
# =====================================================
# Discriminator 第1-3层: 初始特征提取
# =====================================================

print("\n" + "="*80)
print("Discriminator 层级详解")
print("="*80)

print("\n[层1] Conv2d (第一层，不使用BatchNorm)")
print("-" * 80)
print("功能: 提取低级特征（边缘、纹理）")
print()
print("配置:")
print("  输入通道: 1 (灰度图)")
print("  输出通道: 64 (d_features)")
print("  卷积核大小: 4×4")
print("  步长 (stride): 2 (下采样)")
print("  填充 (padding): 1")
print("  bias: True (第一层使用bias)")
print()
print("形状计算:")
print("  H_out = floor((28 + 2×1 - 4) / 2) + 1")
print("  H_out = floor((28 - 2) / 2) + 1 = floor(13) + 1 = 14")
print()
print("参数数量:")
print("  = 4 × 4 × 1 × 64 + 64 = 1,024 + 64 = 1,088")
print()

# 演示
batch_size = 2
input_img = torch.randn(batch_size, 1, 28, 28)  # 模拟真实图像
print(f"输入: 图像")
print(f"  形状: {input_img.shape}")
print(f"  值范围: [{input_img.min():.4f}, {input_img.max():.4f}]")
print()

conv_d_1 = Conv2d(1, 64, 4, 2, 1)
out_conv_d_1 = conv_d_1(input_img)
print(f"  输出形状: {out_conv_d_1.shape}  (batch=2, channels=64, height=14, width=14)")
print(f"  输出示例值范围: [{out_conv_d_1.min():.4f}, {out_conv_d_1.max():.4f}]")
print(f"  总参数: {sum(p.numel() for p in conv_d_1.parameters()):,}")
print()
print("  说明: 28×28 → 14×14，空间尺寸减半，提取64个特征图")
print("  ⚠️  第一层不用BatchNorm，保留输入的原始统计特性")

print("\n[层2] LeakyReLU(0.2)")
print("-" * 80)
print("功能: 非线性激活，允许负梯度")
print()
print("公式: y = x if x > 0 else 0.2 × x")
print()
print("为什么用 LeakyReLU？")
print("  • 相比ReLU，负值不会完全截断")
print("  • 负斜率0.2允许梯度回流")
print("  • 防止'神经元死亡'问题")
print("  • DCGAN论文推荐用于判别器")
print()
leaky_relu = LeakyReLU(0.2)
out_lrelu_d_1 = leaky_relu(out_conv_d_1)
print(f"  输入形状: {out_conv_d_1.shape}")
print(f"  输出形状: {out_lrelu_d_1.shape}")
print(f"  输出示例值范围: [{out_lrelu_d_1.min():.4f}, {out_lrelu_d_1.max():.4f}]")
print()
print("  示例: 输入 [-2, -1, 0, 1, 2]")
print("       输出 [-0.4, -0.2, 0, 1, 2]")

print("\n[层3] Dropout2d(0.3)")
print("-" * 80)
print("功能: 随机丢弃整个特征图通道，防止过拟合")
print()
print("配置:")
print("  dropout概率: 0.3 (30%的通道被随机丢弃)")
print()
print("工作原理:")
print("  • 训练时: 随机将30%的特征图全部置0")
print("  • 测试时: 不做任何丢弃")
print("  • Dropout2d vs Dropout: 丢弃整个通道而不是单个元素")
print()
dropout_d_1 = nn.Dropout2d(0.3)
# 训练模式
dropout_d_1.train()
out_dropout_d_1 = dropout_d_1(out_lrelu_d_1)
print(f"  输入形状: {out_lrelu_d_1.shape}  (64个特征图)")
print(f"  输出形状: {out_dropout_d_1.shape}")
print(f"  参数数量: 0 (无需训练)")
print()
print("  说明: 约 64 × 0.3 = 19 个特征图在训练时被随机设为0")


Discriminator 层级详解

[层1] Conv2d (第一层，不使用BatchNorm)
--------------------------------------------------------------------------------
功能: 提取低级特征

配置:
  输入通道: 1 (灰度图)
  输出通道: 64
  卷积核大小: 4×4
  步长 (stride): 2 (下采样)
  填充 (padding): 1
  bias: False

形状计算:
  H_out = floor((28 + 2×1 - 4) / 2) + 1 = floor(26/2) + 1 = 14

参数数量:
  = 4 × 4 × 1 × 64 = 1,024

  输入形状: torch.Size([2, 1, 28, 28])  (batch=2, channels=1, height=28, width=28)
  输出形状: torch.Size([2, 64, 14, 14])  (batch=2, channels=64, height=14, width=14)
  输出示例值范围: [-2.3443, 2.6417]
  说明: 不使用BatchNorm因为要保留原始特征统计信息

[层2] LeakyReLU激活
--------------------------------------------------------------------------------
功能: 非线性激活，允许小的负值梯度

配置: 负斜率 (negative_slope) = 0.2

公式: y = max(0.2x, x)

  输入形状: torch.Size([2, 64, 14, 14])
  输出形状: torch.Size([2, 64, 14, 14])
  输出示例值范围: [-0.4689, 2.6417]
  说明: 负值也被保留，乘以0.2，这有助于梯度流动


In [13]:
# =====================================================
# Discriminator 第4-7层: 第一个卷积块
# =====================================================

print("\n" + "="*80)
print("Discriminator 第4-7层: 第一个卷积块 (14×14 → 7×7)")
print("="*80)

print("\n[层4] Conv2d")
print("-" * 80)
print("功能: 进一步下采样并增加通道")
print()
print("配置:")
print("  输入通道: 64 (d_features)")
print("  输出通道: 128 (d_features × 2)")
print("  卷积核大小: 4×4")
print("  步长 (stride): 2")
print("  填充 (padding): 1")
print("  bias: True")
print()
print("形状计算:")
print("  H_out = floor((14 + 2×1 - 4) / 2) + 1 = floor(12/2) + 1 = 7")
print()
print("参数数量:")
print("  = 4 × 4 × 64 × 128 + 128 = 131,072 + 128 = 131,200")
print()
conv_d_2 = Conv2d(64, 128, 4, 2, 1)
out_conv_d_2 = conv_d_2(out_dropout_d_1)
print(f"  输入形状: {out_dropout_d_1.shape}  (batch=2, channels=64, height=14, width=14)")
print(f"  输出形状: {out_conv_d_2.shape}  (batch=2, channels=128, height=7, width=7)")
print(f"  输出示例值范围: [{out_conv_d_2.min():.4f}, {out_conv_d_2.max():.4f}]")
print(f"  总参数: {sum(p.numel() for p in conv_d_2.parameters()):,}")

print("\n[层5] BatchNorm2d")
print("-" * 80)
print("功能: 归一化，稳定训练")
print()
print("配置: 通道数 = 128")
print("参数数量: 2 × 128 = 256 (gamma和beta)")
print()
batch_norm_d_1 = BatchNorm2d(128)
out_bn_d_1 = batch_norm_d_1(out_conv_d_2)
print(f"  输入形状: {out_conv_d_2.shape}")
print(f"  输出形状: {out_bn_d_1.shape}")
print(f"  输出示例值范围: [{out_bn_d_1.min():.4f}, {out_bn_d_1.max():.4f}]")
print()
print("  说明: 从第二层开始使用BatchNorm")

print("\n[层6] LeakyReLU(0.2)")
print("-" * 80)
out_lrelu_d_2 = nn.functional.leaky_relu(out_bn_d_1, 0.2)
print(f"  输入形状: {out_bn_d_1.shape}")
print(f"  输出形状: {out_lrelu_d_2.shape}")
print(f"  输出示例值范围: [{out_lrelu_d_2.min():.4f}, {out_lrelu_d_2.max():.4f}]")

print("\n[层7] Dropout2d(0.3)")
print("-" * 80)
print("功能: 继续正则化，防止过拟合")
print()
dropout_d_2 = nn.Dropout2d(0.3)
dropout_d_2.train()
out_dropout_d_2 = dropout_d_2(out_lrelu_d_2)
print(f"  输入形状: {out_lrelu_d_2.shape}  (128个特征图)")
print(f"  输出形状: {out_dropout_d_2.shape}")
print(f"  说明: 约 128 × 0.3 = 38 个特征图被随机丢弃")
print()
print("  当前状态: 14×14 → 7×7，通道 64 → 128")


Discriminator 第3-6层: 第一个残差块

[层3] Conv2d
--------------------------------------------------------------------------------
功能: 进一步下采样并增加通道

配置:
  输入通道: 64
  输出通道: 128
  卷积核大小: 4×4
  步长 (stride): 2
  填充 (padding): 1

形状计算:
  H_out = floor((14 + 2×1 - 4) / 2) + 1 = floor(12/2) + 1 = 7

参数数量:
  = 4 × 4 × 64 × 128 = 131,072

  输入形状: torch.Size([2, 64, 14, 14])  (batch=2, channels=64, height=14, width=14)
  输出形状: torch.Size([2, 128, 7, 7])  (batch=2, channels=128, height=7, width=7)
  输出示例值范围: [-0.8596, 0.9384]

[层4] BatchNorm2d
--------------------------------------------------------------------------------
配置: 通道数 = 128
参数数量: 2 × 128 = 256

  输入形状: torch.Size([2, 128, 7, 7])
  输出形状: torch.Size([2, 128, 7, 7])
  输出示例值范围: [-3.8659, 3.7065]

[层5] LeakyReLU激活
--------------------------------------------------------------------------------
  输入形状: torch.Size([2, 128, 7, 7])
  输出形状: torch.Size([2, 128, 7, 7])
  输出示例值范围: [-0.7732, 3.7065]

[层6] Dropout2d
-----------------------------------------

In [14]:
# =====================================================
# Discriminator 第8-11层: 第二个卷积块
# =====================================================

print("\n" + "="*80)
print("Discriminator 第8-11层: 第二个卷积块 (7×7 → 3×3)")
print("="*80)

print("\n[层8] Conv2d")
print("-" * 80)
print("功能: 继续下采样，提取更抽象的特征")
print()
print("配置:")
print("  输入通道: 128 (d_features × 2)")
print("  输出通道: 256 (d_features × 4)")
print("  卷积核大小: 4×4")
print("  步长 (stride): 2")
print("  填充 (padding): 1")
print()
print("形状计算:")
print("  H_out = floor((7 + 2×1 - 4) / 2) + 1 = floor(5/2) + 1 = 3")
print()
print("参数数量:")
print("  = 4 × 4 × 128 × 256 + 256 = 524,288 + 256 = 524,544")
print()
conv_d_3 = Conv2d(128, 256, 4, 2, 1)
out_conv_d_3 = conv_d_3(out_dropout_d_2)
print(f"  输入形状: {out_dropout_d_2.shape}  (batch=2, channels=128, height=7, width=7)")
print(f"  输出形状: {out_conv_d_3.shape}  (batch=2, channels=256, height=3, width=3)")
print(f"  输出示例值范围: [{out_conv_d_3.min():.4f}, {out_conv_d_3.max():.4f}]")
print(f"  总参数: {sum(p.numel() for p in conv_d_3.parameters()):,}")

print("\n[层9] BatchNorm2d")
print("-" * 80)
print("配置: 通道数 = 256")
print("参数数量: 2 × 256 = 512")
print()
batch_norm_d_2 = BatchNorm2d(256)
out_bn_d_2 = batch_norm_d_2(out_conv_d_3)
print(f"  输入形状: {out_conv_d_3.shape}")
print(f"  输出形状: {out_bn_d_2.shape}")
print(f"  输出示例值范围: [{out_bn_d_2.min():.4f}, {out_bn_d_2.max():.4f}]")

print("\n[层10] LeakyReLU(0.2)")
print("-" * 80)
out_lrelu_d_3 = nn.functional.leaky_relu(out_bn_d_2, 0.2)
print(f"  输入形状: {out_bn_d_2.shape}")
print(f"  输出形状: {out_lrelu_d_3.shape}")
print(f"  输出示例值范围: [{out_lrelu_d_3.min():.4f}, {out_lrelu_d_3.max():.4f}]")

print("\n[层11] Dropout2d(0.3)")
print("-" * 80)
dropout_d_3 = nn.Dropout2d(0.3)
dropout_d_3.train()
out_dropout_d_3 = dropout_d_3(out_lrelu_d_3)
print(f"  输入形状: {out_lrelu_d_3.shape}  (256个特征图)")
print(f"  输出形状: {out_dropout_d_3.shape}")
print(f"  说明: 约 256 × 0.3 = 77 个特征图被随机丢弃")
print()
print("  当前状态: 7×7 → 3×3，通道 128 → 256")


Discriminator 第7-10层: 第二个残差块

[层7] Conv2d
--------------------------------------------------------------------------------
配置:
  输入通道: 128
  输出通道: 256
  卷积核大小: 4×4
  步长 (stride): 2
  填充 (padding): 1

形状计算:
  H_out = floor((7 + 2×1 - 4) / 2) + 1 = floor(5/2) + 1 = 3

参数数量:
  = 4 × 4 × 128 × 256 = 524,288

  输入形状: torch.Size([2, 128, 7, 7])  (batch=2, channels=128, height=7, width=7)
  输出形状: torch.Size([2, 256, 3, 3])  (batch=2, channels=256, height=3, width=3)
  输出示例值范围: [-1.7807, 1.5853]

[层8] BatchNorm2d
--------------------------------------------------------------------------------
  输入形状: torch.Size([2, 256, 3, 3])
  输出形状: torch.Size([2, 256, 3, 3])
  参数数量: 2 × 256 = 512

[层9] LeakyReLU激活
--------------------------------------------------------------------------------
  输入形状: torch.Size([2, 256, 3, 3])
  输出形状: torch.Size([2, 256, 3, 3])

[层10] Dropout2d
--------------------------------------------------------------------------------
  输入形状: torch.Size([2, 256, 3, 3])  (256个特征图)
  

In [15]:
# =====================================================
# Discriminator 第12-17层: 最终特征提取和分类
# =====================================================

print("\n" + "="*80)
print("Discriminator 第12-17层: 最终分类")
print("="*80)

print("\n[层12] Conv2d (最后一层卷积)")
print("-" * 80)
print("功能: 压缩到1×1，提取全局特征")
print()
print("配置:")
print("  输入通道: 256 (d_features × 4)")
print("  输出通道: 512 (d_features × 8)")
print("  卷积核大小: 4×4")
print("  步长 (stride): 2")
print("  填充 (padding): 1")
print()
print("形状计算:")
print("  H_out = floor((3 + 2×1 - 4) / 2) + 1")
print("  H_out = floor(1/2) + 1 = 0 + 1 = 1")
print("  所以 3×3 → 1×1")
print()
print("参数数量:")
print("  = 4 × 4 × 256 × 512 + 512 = 2,097,152 + 512 = 2,097,664")
print()
conv_d_4 = Conv2d(256, 512, 4, 2, 1)
out_conv_d_4 = conv_d_4(out_dropout_d_3)
print(f"  输入形状: {out_dropout_d_3.shape}  (batch=2, channels=256, height=3, width=3)")
print(f"  输出形状: {out_conv_d_4.shape}  (batch=2, channels=512, height=1, width=1)")
print(f"  输出示例值范围: [{out_conv_d_4.min():.4f}, {out_conv_d_4.max():.4f}]")
print(f"  总参数: {sum(p.numel() for p in conv_d_4.parameters()):,}")
print()
print("  ✅ 成功压缩到 1×1 空间尺寸!")

print("\n[层13] BatchNorm2d")
print("-" * 80)
print("配置: 通道数 = 512")
print("参数数量: 2 × 512 = 1,024")
print()
batch_norm_d_3 = BatchNorm2d(512)
out_bn_d_3 = batch_norm_d_3(out_conv_d_4)
print(f"  输入形状: {out_conv_d_4.shape}")
print(f"  输出形状: {out_bn_d_3.shape}")
print(f"  输出示例值范围: [{out_bn_d_3.min():.4f}, {out_bn_d_3.max():.4f}]")

print("\n[层14] LeakyReLU(0.2)")
print("-" * 80)
out_lrelu_d_4 = nn.functional.leaky_relu(out_bn_d_3, 0.2)
print(f"  输入形状: {out_bn_d_3.shape}")
print(f"  输出形状: {out_lrelu_d_4.shape}")
print(f"  输出示例值范围: [{out_lrelu_d_4.min():.4f}, {out_lrelu_d_4.max():.4f}]")
print()
print("  说明: 此时已经是 512个 1×1 的特征图")
print("        每个特征图代表一个全局特征")

print("\n[层15] Flatten (展平)")
print("-" * 80)
print("功能: 准备送入全连接层")
print()
out_flatten = out_lrelu_d_4.view(out_lrelu_d_4.size(0), -1)
print(f"  输入形状: {out_lrelu_d_4.shape}  (batch=2, channels=512, height=1, width=1)")
print(f"  输出形状: {out_flatten.shape}  (batch=2, features=512)")
print(f"  说明: 512×1×1 = 512个全局特征向量")

print("\n[层16] Linear (全连接层)")
print("-" * 80)
print("功能: 输出单个标量，代表真假判别")
print()
print("配置:")
print("  输入特征: 512")
print("  输出特征: 1")
print("  bias: True")
print()
print("参数数量:")
print("  = 512 × 1 + 1 = 513")
print()
fc_d = Linear(512, 1)
out_fc_d = fc_d(out_flatten)
print(f"  输入形状: {out_flatten.shape}")
print(f"  输出形状: {out_fc_d.shape}")
print(f"  输出示例值: {out_fc_d.detach().numpy().flatten()}")
print(f"  输出值范围: [{out_fc_d.min():.4f}, {out_fc_d.max():.4f}]")
print(f"  总参数: {sum(p.numel() for p in fc_d.parameters()):,}")
print()
print("  说明: 这是原始 logits，需要通过 Sigmoid 转换为概率")

print("\n[层17] Sigmoid (输出激活)")
print("-" * 80)
print("功能: 将 logits 转换为概率 [0, 1]")
print()
print("公式: σ(x) = 1 / (1 + e^(-x))")
print()
print("输出含义:")
print("  • 0.0 ~ 0.3: 很可能是假图像 (生成的)")
print("  • 0.4 ~ 0.6: 不确定")
print("  • 0.7 ~ 1.0: 很可能是真图像 (真实的)")
print()
sigmoid = nn.Sigmoid()
out_sigmoid = sigmoid(out_fc_d)
print(f"  输入形状: {out_fc_d.shape}")
print(f"  输出形状: {out_sigmoid.shape}")
print(f"  输入值(logits): {out_fc_d.detach().numpy().flatten()}")
print(f"  输出值(概率): {out_sigmoid.detach().numpy().flatten()}")
print(f"  输出范围: [{out_sigmoid.min():.4f}, {out_sigmoid.max():.4f}] ∈ [0, 1]")
print()
print("  ✓ 判别器完成! 输出真假概率")


Discriminator 第11-15层: 最终分类

[层11] Conv2d (最后一层卷积)
--------------------------------------------------------------------------------
配置:
  输入通道: 256
  输出通道: 512
  卷积核大小: 4×4
  步长 (stride): 2
  填充 (padding): 1

形状计算:
  H_out = floor((3 + 2×1 - 4) / 2) + 1 = floor(1/2) + 1 = 1
  所以 3×3 → 1×1

参数数量:
  = 4 × 4 × 256 × 512 = 2,097,152

  输入形状: torch.Size([2, 256, 3, 3])  (batch=2, channels=256, height=3, width=3)
  输出形状: torch.Size([2, 512, 1, 1])  (batch=2, channels=512, height=?, width=?)
  实际输出形状: torch.Size([2, 512, 1, 1])
  ✅ 输出是 1×1，符合预期！

[层12] BatchNorm2d
--------------------------------------------------------------------------------
  输入形状: torch.Size([2, 512, 1, 1])
  输出形状: torch.Size([2, 512, 1, 1])
  参数数量: 2 × 512 = 1,024

[层13] LeakyReLU激活
--------------------------------------------------------------------------------
  输入形状: torch.Size([2, 512, 1, 1])
  输出形状: torch.Size([2, 512, 1, 1])
  说明: 此时已经是512个1×1的特征图

[层14] Flatten（展平）
------------------------------------------------

---
# 第五部分：Discriminator 总体统计

## Discriminator 的完整路径

In [16]:
# Discriminator完整总结
print("\n" + "="*80)
print("Discriminator 完整汇总 (基于 DCGAN_Clean.ipynb)")
print("="*80)

print("\n架构: Conv2d 下采样 + LeakyReLU + Dropout2d")
print("输入: 图像 (batch_size, 1, 28, 28) ∈ [-1, 1]")
print("输出: 判别概率 (batch_size, 1) ∈ [0, 1]")
print()

print("\n层级映射关系:")
print()
stages_d = [
    ("1. Conv2d 1→64", "(B, 1, 28, 28)", "(B, 64, 14, 14)", 1088),
    ("2. LeakyReLU(0.2)", "(B, 64, 14, 14)", "(B, 64, 14, 14)", 0),
    ("3. Dropout2d(0.3)", "(B, 64, 14, 14)", "(B, 64, 14, 14)", 0),
    ("4. Conv2d 64→128", "(B, 64, 14, 14)", "(B, 128, 7, 7)", 131200),
    ("5. BatchNorm2d", "(B, 128, 7, 7)", "(B, 128, 7, 7)", 256),
    ("6. LeakyReLU(0.2)", "(B, 128, 7, 7)", "(B, 128, 7, 7)", 0),
    ("7. Dropout2d(0.3)", "(B, 128, 7, 7)", "(B, 128, 7, 7)", 0),
    ("8. Conv2d 128→256", "(B, 128, 7, 7)", "(B, 256, 3, 3)", 524544),
    ("9. BatchNorm2d", "(B, 256, 3, 3)", "(B, 256, 3, 3)", 512),
    ("10. LeakyReLU(0.2)", "(B, 256, 3, 3)", "(B, 256, 3, 3)", 0),
    ("11. Dropout2d(0.3)", "(B, 256, 3, 3)", "(B, 256, 3, 3)", 0),
    ("12. Conv2d 256→512", "(B, 256, 3, 3)", "(B, 512, 1, 1)", 2097664),
    ("13. BatchNorm2d", "(B, 512, 1, 1)", "(B, 512, 1, 1)", 1024),
    ("14. LeakyReLU(0.2)", "(B, 512, 1, 1)", "(B, 512, 1, 1)", 0),
    ("15. Flatten", "(B, 512, 1, 1)", "(B, 512)", 0),
    ("16. Linear 512→1", "(B, 512)", "(B, 1) logits", 513),
    ("17. Sigmoid", "(B, 1) logits", "(B, 1) prob ∈ [0,1]", 0),
]

print(f"{'序号':<20} {'输入形状':<25} {'输出形状':<28} {'参数数量':<15}")
print("-" * 90)
total_params_d = 0
for stage, input_shape, output_shape, params in stages_d:
    print(f"{stage:<20} {input_shape:<25} {output_shape:<28} {params:>14,}")
    total_params_d += params

print("-" * 90)
print(f"{'总计':<20} {'':<25} {'':<28} {total_params_d:>14,}")
print()

print("\n关键特征:")
print("  • 输入: 1×28×28 灰度图像 ∈ [-1, 1]")
print("  • 4次下采样: 28×28 → 14×14 → 7×7 → 3×3 → 1×1")
print("  • 通道递增: 1 → 64 → 128 → 256 → 512")
print("  • 激活函数: LeakyReLU(0.2) - 允许负梯度流动")
print("  • 第一层不使用BatchNorm (保留原始统计特性)")
print("  • 中间层使用Dropout2d(0.3) 防止过拟合")
print("  • 最后输出经过Sigmoid，得到概率 [0, 1]")
print(f"  • 总参数数: {total_params_d:,}")
print()

print("\n架构特点:")
print("  ✓ 使用 4×4 卷积核，stride=2 进行下采样")
print("  ✓ 每次下采样都增倍通道数")
print("  ✓ Dropout2d 丢弃整个特征图通道")
print("  ✓ 最终压缩到 512×1×1 全局特征")
print("  ✓ 用 Linear + Sigmoid 输出真假概率")
print()

print("\n数据流总结:")
print("  [输入图像] 1×28×28")
print("     ↓ Conv+LeakyReLU+Dropout (无BN)")
print("  [特征1] 64×14×14")
print("     ↓ Conv+BN+LeakyReLU+Dropout")
print("  [特征2] 128×7×7")
print("     ↓ Conv+BN+LeakyReLU+Dropout")
print("  [特征3] 256×3×3")
print("     ↓ Conv+BN+LeakyReLU")
print("  [全局特征] 512×1×1")
print("     ↓ Flatten+Linear+Sigmoid")
print("  [输出] 概率 ∈ [0,1]")
print("         (0=假, 1=真)")


Discriminator 完整汇总

输入: 图像 (batch_size, 1, 28, 28) ∈ [-1, 1]
输出: 判别值 (batch_size, 1) ∈ ℝ (logits)


层级映射关系:

序号                   输入形状                      输出形状                      参数数量           
-------------------------------------------------------------------------------------
1. Conv2d 1→64       (B, 1, 28, 28)            (B, 64, 14, 14)                    1,024
2. LeakyReLU         (B, 64, 14, 14)           (B, 64, 14, 14)                        0
3. Conv2d 64→128     (B, 64, 14, 14)           (B, 128, 7, 7)                   131,072
4. BatchNorm2d       (B, 128, 7, 7)            (B, 128, 7, 7)                       256
5. LeakyReLU         (B, 128, 7, 7)            (B, 128, 7, 7)                         0
6. Dropout2d         (B, 128, 7, 7)            (B, 128, 7, 7)                         0
7. Conv2d 128→256    (B, 128, 7, 7)            (B, 256, 3, 3)                   524,288
8. BatchNorm2d       (B, 256, 3, 3)            (B, 256, 3, 3)                       512
9. LeakyReL

In [17]:
# =====================================================
# 完整Discriminator模拟 (基于DCGAN_Clean架构)
# =====================================================

print("\n" + "="*80)
print("完整Discriminator前向传播示例")
print("="*80)

# 配置
class SimpleConfigD:
    channels = 1
    d_features = 64
    dropout = 0.3

cfg_d = SimpleConfigD()

# 构建完整的Discriminator
class SimpleDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            # 第一层 (不用BN)
            nn.Conv2d(cfg_d.channels, cfg_d.d_features, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(cfg_d.dropout),
            
            # 第一个卷积块
            nn.Conv2d(cfg_d.d_features, cfg_d.d_features * 2, 4, 2, 1),
            nn.BatchNorm2d(cfg_d.d_features * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(cfg_d.dropout),
            
            # 第二个卷积块
            nn.Conv2d(cfg_d.d_features * 2, cfg_d.d_features * 4, 4, 2, 1),
            nn.BatchNorm2d(cfg_d.d_features * 4),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(cfg_d.dropout),
            
            # 第三个卷积块
            nn.Conv2d(cfg_d.d_features * 4, cfg_d.d_features * 8, 4, 2, 1),
            nn.BatchNorm2d(cfg_d.d_features * 8),
            nn.LeakyReLU(0.2),
        )
        self.fc = nn.Linear(cfg_d.d_features * 8 * 1 * 1, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.main(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return self.sigmoid(x)

# 实例化
D_demo = SimpleDiscriminator()
total_params_d = sum(p.numel() for p in D_demo.parameters())

print(f"\n构建Discriminator成功!")
print(f"总参数数: {total_params_d:,}")
print()

# 测试两种输入
print("测试示例:")
print("-" * 80)

# 真实图像 (模拟)
real_img = torch.randn(2, 1, 28, 28) * 0.5  # 接近真实数据分布
print(f"输入1: 真实图像 (模拟)")
print(f"  形状: {real_img.shape}")
print(f"  值范围: [{real_img.min():.4f}, {real_img.max():.4f}]")

# 假图像 (随机噪声)
fake_img = torch.randn(2, 1, 28, 28)
print(f"\n输入2: 假图像 (随机噪声)")
print(f"  形状: {fake_img.shape}")
print(f"  值范围: [{fake_img.min():.4f}, {fake_img.max():.4f}]")
print()

# 前向传播
with torch.no_grad():
    D_demo.eval()
    real_pred = D_demo(real_img)
    fake_pred = D_demo(fake_img)

print(f"输出: 判别概率")
print(f"  真实图像判别结果: {real_pred.squeeze().numpy()}")
print(f"    → 平均: {real_pred.mean():.4f} (期望接近1.0)")
print()
print(f"  假图像判别结果: {fake_pred.squeeze().numpy()}")
print(f"    → 平均: {fake_pred.mean():.4f} (期望接近0.0)")
print()

print("逐层形状变化追踪:")
print("-" * 80)

# 手动追踪每一步
with torch.no_grad():
    x = real_img
    print(f"0. 输入:           {x.shape}")
    
    layer_idx = 1
    for i, layer in enumerate(D_demo.main):
        x = layer(x)
        layer_name = layer.__class__.__name__
        print(f"{layer_idx}. {layer_name:<16} → {x.shape}")
        layer_idx += 1
    
    x = x.view(x.size(0), -1)
    print(f"{layer_idx}. Flatten         → {x.shape}")
    layer_idx += 1
    
    x = D_demo.fc(x)
    print(f"{layer_idx}. Linear          → {x.shape}")
    layer_idx += 1
    
    x = D_demo.sigmoid(x)
    print(f"{layer_idx}. Sigmoid         → {x.shape}")

print()
print("="*80)
print("✓ Discriminator 完整流程演示完成!")
print("="*80)
print()
print("总结:")
print("  • 从 28×28 图像 开始")
print("  • 经过 17层网络 处理")
print("  • 最终输出 真假概率 [0, 1]")
print("  • 使用 LeakyReLU 保持梯度流动")
print("  • 使用 Dropout2d 防止过拟合")
print(f"  • 总共使用 {total_params_d:,} 个可训练参数")


完整Discriminator前向传播示例

构建Discriminator成功!
总参数数: 2,756,801

测试示例:
--------------------------------------------------------------------------------
输入1: 真实图像 (模拟)
  形状: torch.Size([2, 1, 28, 28])
  值范围: [-1.5905, 1.5163]

输入2: 假图像 (随机噪声)
  形状: torch.Size([2, 1, 28, 28])
  值范围: [-3.2538, 2.6443]

输出: 判别概率
  真实图像判别结果: [0.50446284 0.50305617]
    → 平均: 0.5038 (期望接近1.0)

  假图像判别结果: [0.5035321 0.5028545]
    → 平均: 0.5032 (期望接近0.0)

逐层形状变化追踪:
--------------------------------------------------------------------------------
0. 输入:           torch.Size([2, 1, 28, 28])
1. Conv2d           → torch.Size([2, 64, 14, 14])
2. LeakyReLU        → torch.Size([2, 64, 14, 14])
3. Dropout2d        → torch.Size([2, 64, 14, 14])
4. Conv2d           → torch.Size([2, 128, 7, 7])
5. BatchNorm2d      → torch.Size([2, 128, 7, 7])
6. LeakyReLU        → torch.Size([2, 128, 7, 7])
7. Dropout2d        → torch.Size([2, 128, 7, 7])
8. Conv2d           → torch.Size([2, 256, 3, 3])
9. BatchNorm2d      → torch.Size([2, 256

---
# 第六部分：完整数据流可视化

---
# 第七部分：总结表格

In [18]:
# 创建总结表
import pandas as pd

print("\n" + "="*100)
print("Generator vs Discriminator 对比")
print("="*100)
print()

comparison_data = {
    '特征': [
        '功能',
        '输入',
        '输出',
        '主要操作',
        '通道变化',
        '空间维度',
        '激活函数',
        'BatchNorm位置',
        'Dropout',
        '最后激活',
        '总参数数',
        '训练目标',
    ],
    'Generator': [
        '生成真实的图像',
        '隐向量 (100,)',
        '图像 (1, 28, 28)',
        'Upsample + Conv2d (上采样)',
        '128→64→1',
        '7×7 → 14×14 → 28×28',
        'ReLU',
        '线性层后和所有卷积层前',
        '无',
        'Tanh ([-1, 1])',
        '849,793',
        '最小化判别器的识别能力',
    ],
    'Discriminator': [
        '区分真假图像',
        '图像 (1, 28, 28)',
        '判别概率 (0~1)',
        'Conv2d (下采样)',
        '1→64→128→256→512',
        '28×28 → 14×14 → 7×7 → 3×3 → 1×1',
        'LeakyReLU(0.2)',
        '第一层无，其他层有',
        '有 (0.3概率)',
        'Sigmoid ([0, 1])',
        '2,755,801',
        '最大化判别能力',
    ],
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))
print()

print("\n" + "="*100)
print("参数统计对比")
print("="*100)
print()

# Generator 参数详细分解
print("Generator 参数详细 (849,793):")
print("  • Linear (100→6272):         627,200 + 6,272 (bias)   = 633,472")
print("  • BatchNorm2d(128):          256")
print("  • Conv2d(128→64, 3×3):       73,728 + 64 (bias)       = 73,792")
print("  • BatchNorm2d(64):           128")
print("  • Conv2d(64→64, 3×3):        36,864 + 64 (bias)       = 36,928")
print("  • BatchNorm2d(64):           128")
print("  • Conv2d(64→1, 3×3):         576 + 1 (bias)           = 577")
print(f"  总计: 849,793")
print()

# Discriminator 参数详细分解
print("Discriminator 参数详细 (2,755,801):")
print("  • Conv2d(1→64, 4×4):         1,024 + 64 (bias)        = 1,088")
print("  • Conv2d(64→128, 4×4):       131,072 + 128 (bias)     = 131,200")
print("  • BatchNorm2d(128):          256")
print("  • Conv2d(128→256, 4×4):      524,288 + 256 (bias)     = 524,544")
print("  • BatchNorm2d(256):          512")
print("  • Conv2d(256→512, 4×4):      2,097,152 + 512 (bias)   = 2,097,664")
print("  • BatchNorm2d(512):          1,024")
print("  • Linear(512→1):             512 + 1 (bias)           = 513")
print(f"  总计: 2,755,801")
print()

# 对比统计
total_params = 849793 + 2755801
g_ratio = 849793 / 2755801
d_ratio = 2755801 / 849793

print(f"DCGAN 总参数数: {total_params:,}")
print(f"Generator 占比: {849793/total_params*100:.1f}%")
print(f"Discriminator 占比: {2755801/total_params*100:.1f}%")
print(f"Discriminator/Generator 参数比: {d_ratio:.2f}:1")
print()

print("架构对比说明:")
print("  ✅ Generator 使用 Upsample+Conv 代替 ConvTranspose2d")
print("     → 避免棋盘效应 (checkerboard artifacts)")
print("     → 参数量大幅减少: 849,793 vs 旧架构 3,572,864")
print("     → 生成质量更稳定")
print()
print("  ✅ Discriminator 使用 stride=2 实现下采样")
print("     → 第一层不使用 BatchNorm (保留原始特征)")
print("     → 使用 LeakyReLU(0.2) 保持梯度流动")
print("     → 使用 Dropout2d(0.3) 防止过拟合")
print("     → 最后使用 Sigmoid 输出概率 [0, 1]")
print()
print("  ⚖️ 参数不对称性:")
print("     → Discriminator 参数是 Generator 的 3.24 倍")
print("     → 这是正常的: Discriminator 需要更强的判别能力")
print("     → Generator 通过对抗训练逐步提升生成质量")



Generator vs Discriminator 对比

         特征               Generator                   Discriminator
         功能                 生成真实的图像                          区分真假图像
         输入              隐向量 (100,)                  图像 (1, 28, 28)
         输出          图像 (1, 28, 28)                      判别概率 (0~1)
       主要操作 Upsample + Conv2d (上采样)                    Conv2d (下采样)
       通道变化                128→64→1                1→64→128→256→512
       空间维度     7×7 → 14×14 → 28×28 28×28 → 14×14 → 7×7 → 3×3 → 1×1
       激活函数                    ReLU                  LeakyReLU(0.2)
BatchNorm位置             线性层后和所有卷积层前                       第一层无，其他层有
    Dropout                       无                       有 (0.3概率)
       最后激活          Tanh ([-1, 1])                Sigmoid ([0, 1])
       总参数数                 849,793                       2,755,801
       训练目标             最小化判别器的识别能力                         最大化判别能力


参数统计对比

Generator 参数详细 (849,793):
  • Linear (100→6272):         627,200 + 6,272 (

---
# 第八部分：数学公式汇总

## 关键的形状变换公式

### 1. Conv2d输出高度/宽度
$$H_{out} = \left\lfloor \frac{H_{in} + 2P - K}{S} \right\rfloor + 1$$

**例子** (Discriminator第一层):
- 输入: 28×28
- 核: 4, 填充: 0, 步长: 2
$$H = \left\lfloor \frac{28 + 2×0 - 4}{2} \right\rfloor + 1 = \left\lfloor \frac{24}{2} \right\rfloor + 1 = 13$$

等等，实际输出是14×14，说明使用了 padding=1:
$$H = \left\lfloor \frac{28 + 2×1 - 4}{2} \right\rfloor + 1 = \left\lfloor \frac{26}{2} \right\rfloor + 1 = 14$$

---

### 2. Upsample (双线性插值) 输出高度/宽度

$$H_{out} = H_{in} \times scale\_factor$$

**例子** (Generator第一个Upsample):
- 输入: 7×7
- scale_factor: 2
$$H = 7 × 2 = 14$$

**双线性插值原理**:
对于每个输出像素 $(x', y')$，计算其在原图的坐标 $(x, y)$：
$$x = \frac{x' \times W_{in}}{W_{out}}, \quad y = \frac{y' \times H_{in}}{H_{out}}$$

然后对周围4个像素进行加权平均：
$$I(x', y') = \sum_{i,j} w_{ij} \cdot I(x_i, y_j)$$

其中权重 $w_{ij}$ 由距离决定，越近权重越大。

---

### 3. ConvTranspose2d输出高度/宽度 (参考，当前架构未使用)

$$H_{out} = (H_{in} - 1) × S - 2P + K$$

**例子** (假设的ConvTranspose2d层):
- 输入: 7×7
- 核: 4, 填充: 1, 步长: 2
$$H = (7 - 1) × 2 - 2×1 + 4 = 6×2 - 2 + 4 = 14$$

**为什么不用ConvTranspose2d？**
- 重叠卷积导致棋盘效应
- 参数量大 (例: 128→128 需要 262,144 参数)
- Upsample+Conv 更高效 (128→64 只需 73,792 参数)

---

### 4. BatchNorm公式

$$y = \gamma \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

其中：
- $\mu_B, \sigma_B^2$: batch的均值和方差
- $\gamma, \beta$: 可学习的缩放和偏移参数 (每个通道2个参数)
- $\epsilon$: 小常数(如1e-5)，防止除零

**参数数量**: 
- BatchNorm2d(C): $2 \times C$ 个参数
- 例: BatchNorm2d(128) = 256 个参数

---

### 5. LeakyReLU公式

$$y = \begin{cases} x & \text{if } x > 0 \\ \alpha x & \text{if } x \leq 0 \end{cases}$$

其中 $\alpha = 0.2$ (在Discriminator中)

**为什么用LeakyReLU？**
- ReLU: $y = max(0, x)$，负值梯度为0
- LeakyReLU: 负值梯度为 $\alpha = 0.2$
- 在对抗训练中更稳定，避免"死神经元"

---

### 6. Tanh激活

$$y = \tanh(x) = \frac{e^{2x} - 1}{e^{2x} + 1} = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

输出范围: $(-1, 1)$

**为什么Generator用Tanh？**
- 训练数据归一化到 [-1, 1]
- 输出与数据分布一致
- 配合 Normalize([0.5], [0.5]) 使用

---

### 7. Sigmoid激活

$$y = \sigma(x) = \frac{1}{1 + e^{-x}}$$

输出范围: $(0, 1)$

**Discriminator使用Sigmoid输出概率**:
- $\sigma(x) \approx 1$: 判断为真实图像
- $\sigma(x) \approx 0$: 判断为假图像
- $\sigma(x) \approx 0.5$: 不确定

---

### 8. BCELoss (Binary Cross Entropy)

$$L = -[y \log(\hat{y}) + (1-y) \log(1-\hat{y})]$$

其中：
- $y$: 目标标签 (真实=1, 假=0)
- $\hat{y} = \sigma(x)$: Sigmoid输出的概率

**标签平滑 (Label Smoothing)**:
- 真实标签: 0.9 而不是 1.0
- 假标签: 0.1 而不是 0.0
- 防止过拟合，提高训练稳定性

$$L_{real} = -[0.9 \log(\hat{y}) + 0.1 \log(1-\hat{y})]$$
$$L_{fake} = -[0.1 \log(\hat{y}) + 0.9 \log(1-\hat{y})]$$


---
# 第九部分：问题解答

## Q1: 为什么Generator参数相对较少?

**A:** 当前架构使用 Upsample+Conv 代替 ConvTranspose2d，大幅减少参数：
- 线性层: 100 → 6272 (627,200 参数，用于映射到初始特征)
- Upsample: 确定性上采样，**无参数**
- Conv2d层: 只需学习特征提取，参数少
- 总计 849,793 参数 (vs 旧架构 3.5M)

**参数效率对比**:
- 旧架构 (ConvTranspose2d): 3,572,864 参数
- 新架构 (Upsample+Conv): 849,793 参数
- 减少了 **76.2%** 的参数量

**为什么参数少还能工作？**
- Upsample 使用固定算法 (双线性插值)，不需要学习
- Conv2d 只需学习特征提取模式，更专注
- 避免了 ConvTranspose2d 的参数冗余
- 实践证明: 参数少，训练快，质量好！

---

## Q2: 为什么Discriminator第一层不用BatchNorm?

**A:** 这是DCGAN的一个重要设计原则：
- BatchNorm会改变输入图像的统计特性
- 第一层需要直接观察原始图像特征来区分真假
- 去掉第一层的BN能保留更多的鉴别信息

---

## Q3: 为什么Generator的最后是Tanh而不是Sigmoid?

**A:** 
- Tanh输出范围是[-1, 1]
- 训练数据也被归一化到[-1, 1]: `Normalize([0.5], [0.5])`
- 这样输出和输入的分布一致，有利于训练
- Sigmoid输出是[0, 1]，不匹配

---

## Q4: Upsample+Conv 和 ConvTranspose2d 的区别?

**A: 这是当前架构的关键改进！**

**ConvTranspose2d (旧方法)**:
```
特点:
  • 参数化的上采样，权重可学习
  • 单步完成上采样和卷积
  • 容易产生棋盘效应 (checkerboard artifacts)
  • 参数量大 (例: 128→128 的 4×4 核 = 262,144 参数)
```

**Upsample + Conv2d (新方法)**:
```
特点:
  • 两步操作: 先上采样(无参数) → 再卷积(有参数)
  • Upsample 使用双线性插值，确定性，无参数
  • Conv2d 只需学习特征提取，参数少
  • 避免棋盘效应，生成质量更平滑
  • 参数量小 (例: 128→64 的 3×3 核 = 73,792 参数)
```

**为什么 Upsample+Conv 更好？**
1. **避免棋盘效应**: ConvTranspose2d 的重叠卷积会导致某些像素被计算多次
2. **参数效率**: 分离上采样和特征学习，减少冗余参数
3. **训练稳定**: 双线性插值提供平滑的上采样，梯度更稳定
4. **质量更好**: 实践中生成的图像更清晰，伪影更少

**实际对比**:
- Generator 总参数: 849,793 (新) vs 3,572,864 (旧)
- 减少 76.2% 参数量
- 训练速度更快
- 生成质量更好

---

## Q5: 为什么Discriminator使用LeakyReLU而不是ReLU?

**A:**
- LeakyReLU允许负值通过(乘以0.2)，保留更多梯度信息
- ReLU会完全抹掉负值，可能导致梯度消失
- 在对抗训练中，LeakyReLU更稳定

---

## Q6: Dropout在Generator中为什么没有?

**A:**
- Generator需要一致的输出来欺骗Discriminator
- Dropout会在训练和推理时产生不同的输出
- Discriminator用Dropout来增加鲁棒性

---

## Q7: Discriminator为什么使用Sigmoid输出而不是logits?

**A:**
- 当前架构使用 **Sigmoid** 输出概率 [0, 1]
- 配合 BCELoss (Binary Cross Entropy Loss)
- 优点:
  * 输出直观，可直接解释为"真实概率"
  * 便于监控训练过程
  * 代码更清晰
  
**注意**: 有些实现使用 BCEWithLogitsLoss (不需要Sigmoid)，但当前架构明确使用 Sigmoid + BCELoss 的组合。
